In [1]:
# 0. import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import RidgeCV, Ridge, LogisticRegressionCV, LogisticRegression
import os
%matplotlib inline
from matplotlib.ticker import MaxNLocator
import matplotlib.dates as mdates
import datetime
import json
import statsmodels.api as sm
from patsy import dmatrix
from pathlib import Path


In [2]:
# read data
PROJECT_ROOT = Path("/Users/xueqingliu/Harvard University Dropbox/Liu Xueqing/ADAPR-MRT-Testbed")
COMBINED_DIR = Path("/Users/xueqingliu/Harvard University Dropbox/Liu Xueqing/ADAPT_MRT/rawdata/_combined")
WORK_DIR = PROJECT_ROOT / "env_para_vanilla"
WORK_DIR.mkdir(parents=True, exist_ok=True)

df_merged = pd.read_csv(COMBINED_DIR / "df_merged.csv")

# Aliases for later cells that use `folder` / `work_folder` (Path so `/` joins work)
folder = COMBINED_DIR
work_folder = Path(WORK_DIR)

df_fit = pd.read_csv(folder / 'df_fit.csv')

file_params_env_prefix = str(work_folder / 'params_env_')
file_pred_prefix = str(work_folder / 'pred_')
file_user_ids = str(work_folder / 'user_ids.txt')

In [3]:
print(df_fit.columns)
print(df_fit.shape)
# print(df_fit.CAE_avg_lastweek_norm[50:100])

Index(['ParticipantIdentifier', 'Date', 'DecisionTime', 'week', 'day', 'dow',
       'is_weekend', 'WalkingSuggestion', 'Interacted_walk',
       'Interacted_7d_walk', 'SalienceMessage', 'Interacted_salience',
       'Interacted_7d_salience', 'yesterday_SalienceMessage',
       'yesterday_Interacted', 'yesterday_Interacted_7d', 'planning_prompt',
       'yesterday_planning_prompt', '4hour_step', 'EMA_StepCount',
       'YesterdayStepCount', 'prior2hour_step', 'RecordedPhysicalActivity',
       'Previous7DaysRPA', 'DayWearing', 'nextday_wearing',
       'past7days_daywearing', 'morning_wearing', 'DailyPageviewCount',
       'Past7DaysPageviewEMA', 'TomorrowPageviewCount', 'HourlyPageviewCount',
       'week_present', 'week_present_lastweek', 'daily_present',
       'daily_present_yesterday', 'affective_reflection', 'anticipated_affect',
       'affective_reflection_yesterday', 'anticipated_affect_yesterday',
       'CAE_avg', 'CAE_avg_lastweek', 'CAE_short_avg', 'recent_burden',
       

In [4]:
print(df_fit)
print(df_fit.CAE_avg_lastweek_norm[0:50])

      ParticipantIdentifier        Date  DecisionTime  week  day  dow  \
0                       118  2025-09-14             0     0    0    7   
1                       118  2025-09-14             1     0    0    7   
2                       118  2025-09-15             0     0    1    1   
3                       118  2025-09-15             1     0    1    1   
4                       118  2025-09-16             0     0    2    2   
...                     ...         ...           ...   ...  ...  ...   
1865                    225  2026-03-07             1    11   82    6   
1866                    225  2026-03-08             0    11   83    7   
1867                    225  2026-03-08             1    11   83    7   
1868                    225  2026-03-09             0    12   84    1   
1869                    225  2026-03-09             1    12   84    1   

      is_weekend  WalkingSuggestion  Interacted_walk  Interacted_7d_walk  ...  \
0              1                0.0       

In [5]:
# Define the perceived utility (this is the ground truth)
w_1 = 0.2
w_2 = 0.5
w_3 = 0.1
w_4 = 0.1
w_5 = 0.1

# change it from 0-7 to 1-8
df_fit['Exp-tool-1'] = df_fit['Exp-tool-1'] + 1
df_fit['Exp-tool-2'] = df_fit['Exp-tool-2'] + 1

# Per week: sums over distinct days (daily_present; DayWearing duplicated across DecisionTime)
_weekly_daily = (
    df_fit.assign(_d=pd.to_datetime(df_fit['Date']))
    .groupby(['ParticipantIdentifier', 'week', '_d'], as_index=False)[['daily_present', 'DayWearing', 'DailyPageviewCount']]
    .first()
    .groupby(['ParticipantIdentifier', 'week'])[['daily_present', 'DayWearing', 'DailyPageviewCount']]
    .sum()
    .rename(columns={'daily_present': 'weekly_daily_present_sum', 'DayWearing': 'weekly_daily_wearing_sum', 'DailyPageviewCount': 'weekly_daily_pageview_sum'})
    .reset_index()
)

df_fit = df_fit.merge(_weekly_daily, on=['ParticipantIdentifier', 'week'], how='left')



# min–max to [0, 1]: (x - min) / (max - min); constant column → 0
_pv = df_fit['weekly_daily_pageview_sum']
_lo, _hi = _pv.min(), _pv.max()
df_fit['weekly_daily_pageview_sum'] = (_pv - _lo) / (_hi - _lo) if _hi > _lo else 0.0

print(df_fit.weekly_daily_pageview_sum.unique())
df_fit['perceived_utility'] = (
    w_1 * df_fit['week_present']
    + 0.5 * w_2 * df_fit['week_present'] * (df_fit['Exp-tool-1'] + df_fit['Exp-tool-2'])
    + (w_3 / 7) * df_fit['weekly_daily_present_sum']
    + (w_4 / 7) * df_fit['weekly_daily_wearing_sum']
    + (w_5) * df_fit['weekly_daily_pageview_sum']
)

# add perceived utility last week
df_fit['perceived_utility_lastweek'] = df_fit['perceived_utility'].shift(1)


[0.78761062 0.87610619 0.74336283 0.53097345 0.79646018 0.59292035
 0.67256637 0.44247788 0.32743363 0.21238938 0.05309735 0.31858407
 0.30973451 0.23893805 0.28318584 0.19469027 0.14159292 0.2920354
 0.22123894 0.2300885  0.37168142 0.30088496 0.02654867 0.65486726
 0.38938053 0.4159292  0.26548673 0.38053097 0.25663717 0.3539823
 0.00884956 0.15044248 0.24778761 0.18584071 0.12389381 0.17699115
 0.11504425 0.         0.53982301 0.36283186 0.15929204 0.20353982
 0.09734513 0.0619469  0.01769912 0.46902655 0.47787611 0.49557522
 0.39823009 0.43362832 0.62831858 0.33628319 0.04424779 0.03539823
 0.08849558 0.10619469 1.         0.5840708  0.54867257 0.42477876
 0.34513274 0.13274336 0.07964602 0.6460177  0.48672566 0.46017699
 0.72566372 0.51327434 0.16814159]


In [6]:
# Lag-1 within each participant (global [:-1] misaligns rows and crosses participants)
_sort = ['ParticipantIdentifier', 'Date', 'DecisionTime']
df_fit = df_fit.sort_values(_sort, na_position='last').reset_index(drop=True)
df_fit['FourSC_lag1'] = df_fit.groupby('ParticipantIdentifier', sort=False)['4hour_step_norm'].shift(1)
df_fit['anticipated_affect_lag1'] = df_fit.groupby('ParticipantIdentifier', sort=False)['anticipated_affect_norm'].shift(2)
df_fit['recorded_physical_activity_lag1'] = df_fit.groupby('ParticipantIdentifier', sort=False)['RecordedPhysicalActivity'].shift(2)
df_fit['dow_norm_lag1'] = df_fit.groupby('ParticipantIdentifier', sort=False)['dow_norm'].shift(2)
df_fit['WalkingSuggestion_lag1'] = df_fit.groupby('ParticipantIdentifier', sort=False)['WalkingSuggestion'].shift(2)
df_fit['yesterday_pageview_count'] = df_fit.groupby('ParticipantIdentifier', sort=False)['DailyPageviewCount_norm'].shift(2)
df_fit['yesterday_fitbitwearing'] = df_fit.groupby('ParticipantIdentifier', sort=False)['DayWearing'].shift(2)
df_fit['yesterday_fitbitwearing_morning'] = df_fit.groupby('ParticipantIdentifier', sort=False)['morning_wearing'].shift(2)
df_fit['yesterday_present'] = df_fit.groupby('ParticipantIdentifier', sort=False)['daily_present'].shift(2)
df_fit['hourly_pageview_count_lag1'] = df_fit.groupby('ParticipantIdentifier', sort=False)['HourlyPageviewCount_norm'].shift(1)
df_fit['yesterday_hourly_pageview_count'] = df_fit.groupby('ParticipantIdentifier', sort=False)['HourlyPageviewCount_norm'].shift(2)

In [7]:
# user 141, 143 does not have week 0, change week 0 to week 1
# change all participants' who have more than 2 week 0 to week 1
for userid in df_fit['ParticipantIdentifier'].unique():
    vc = df_fit.loc[df_fit['ParticipantIdentifier'] == userid, 'week'].value_counts()
    if vc.get(0, 0) > 2:
        m = df_fit['ParticipantIdentifier'] == userid
        df_fit.loc[m, 'week'] = df_fit.loc[m, 'week'] + 1

In [8]:
# remove week 0 and week 1 data
df_fit = df_fit[df_fit['week'] > 1]

# turn week 2 into week 1
df_fit['week'] = df_fit['week'] - 1

# remove week 12
df_fit = df_fit[df_fit['week'] < 12]

In [ ]:
df_fit.to_csv(folder / "df_fit_11week.csv")

In [ ]:
# import warnings
# from sklearn.exceptions import UndefinedMetricWarning

alpha_l2_list = [0.2, 0.5, 1, 2, 5]
alpha_lap_list = [0.5, 1, 2, 5]
ncv = 5
seed = 2026
alpha_cv = []
alpha_cv_std = []
dat_user_all = []

userid_all = df_fit['ParticipantIdentifier'].unique()
theta_pageview_list = []
theta_fitbitwearing_list = []
theta_eodcomplete_list = []
for i, userid in enumerate(userid_all):
    dat_user = df_fit[df_fit['ParticipantIdentifier'] == userid].copy()
    dat_user = dat_user.sort_values(['Date', 'DecisionTime'], na_position='last').reset_index(drop=True)

    # fill in initial values (last week's affective association, and perceived utility)
    # set the first 0-13 days to 0
    # if len(dat_user) > 0:
    #     dat_user.loc[dat_user.index[:14], 'week_present_lastweek'] = 0
    #     dat_user.loc[dat_user.index[:14], 'CAE_avg_lastweek_norm'] = 0
    #     dat_user.loc[dat_user.index[:14], 'perceived_utility_lastweek_norm'] = 0
    #     dat_user.loc[dat_user.index[:2], 'view_status_lastdecision'] = 0

    # extract the response
    # M^Y_{w,d,t}
    fourSC = dat_user['4hour_step_norm'].to_numpy()
    fourSC_lag1 = dat_user['FourSC_lag1'].to_numpy()


    # M^Y_{w,d}
    anticipated_affect = dat_user['anticipated_affect_norm'].to_numpy()
    anticipated_affect_yesterday = dat_user['anticipated_affect_yesterday_norm'].to_numpy()
    anticipated_affect_lag1 = dat_user['anticipated_affect_lag1'].to_numpy()

    # M^E_{w,d,t}
    HourlyPageviewCount = dat_user['HourlyPageviewCount_norm'].to_numpy()
    
    # M^E_{w,d}
    # TomorrowPageviewCount_norm = dat_user['TomorrowPageviewCount_norm'].to_numpy()
    nextday_wearing = dat_user['nextday_wearing'].to_numpy()
    daily_present = dat_user['daily_present'].to_numpy()

    # Y_w: CAE
    CAE_avg = dat_user['CAE_avg_norm'].to_numpy()
    # print(userid, CAE_avg_norm.shape)
    CAE_avg_lastweek = dat_user['CAE_avg_lastweek_norm'].to_numpy()

    # tilde Y_w
    CAE_short_avg = dat_user['CAE_short_avg_norm'].to_numpy()

    # E_w
    perceived_utility = dat_user['perceived_utility'].to_numpy()
    perceived_utility_lastweek = dat_user['perceived_utility_lastweek'].to_numpy()

    # extract the predictors
    Intercept = np.ones(len(fourSC))
    yesterday_step_count = dat_user['YesterdayStepCount_norm'].to_numpy()
    seven_day_step_count_avg = dat_user['EMA_StepCount_norm'].to_numpy()
    prior2hour_step_count = dat_user['prior2hour_step_norm'].to_numpy()
    Previous7DaysRPA = dat_user['Previous7DaysRPA'].to_numpy()
    recent_burden = dat_user['recent_burden_norm'].to_numpy()
    recorded_physical_activity = dat_user['RecordedPhysicalActivity'].to_numpy()
    recorded_physical_activity_lag1 = dat_user['recorded_physical_activity_lag1'].to_numpy()
    
    # yesterday_pageview_count = dat_user['yesterday_pageview_count_norm'].to_numpy()
    DailyPageviewCount = dat_user['DailyPageviewCount_norm'].to_numpy()
    yesterday_pageview_count = dat_user['yesterday_pageview_count'].to_numpy()
    seven_day_pageview_count = dat_user['Past7DaysPageviewEMA_norm'].to_numpy()
    HourlyPageviewCount = dat_user['HourlyPageviewCount_norm'].to_numpy()
    hourly_pageview_count_lag1 = dat_user['hourly_pageview_count_lag1'].to_numpy()
    yesterday_hourly_pageview_count = dat_user['yesterday_hourly_pageview_count'].to_numpy()
    DayWearing = dat_user['DayWearing'].to_numpy()
    yesterday_fitbitwearing = dat_user['yesterday_fitbitwearing'].to_numpy()
    fitbitwearing_morning = dat_user['morning_wearing'].to_numpy()
    yesterday_fitbitwearing_morning = dat_user['yesterday_fitbitwearing_morning'].to_numpy()
    past7days_daywearing = dat_user['past7days_daywearing'].to_numpy()
    Interacted_7d_walk = dat_user['Interacted_7d_walk'].to_numpy()
    Interacted_7d_salience = dat_user['Interacted_7d_salience'].to_numpy()
    yesterday_present = dat_user['yesterday_present'].to_numpy()
    daily_present = dat_user['daily_present'].to_numpy()

    affective_reflection = dat_user['affective_reflection_norm'].to_numpy()
    affective_reflection_yesterday = dat_user['affective_reflection_yesterday_norm'].to_numpy()

    week_present = dat_user['week_present'].to_numpy()
    week_present_lastweek = dat_user['week_present_lastweek'].to_numpy()
    
    WalkingSuggestion = dat_user['WalkingSuggestion'].to_numpy()
    WalkingSuggestion_lag1 = dat_user['WalkingSuggestion_lag1'].to_numpy()
    yesterday_SalienceMessage = dat_user['yesterday_SalienceMessage'].to_numpy()
    yesterday_planning_prompt = dat_user['yesterday_planning_prompt'].to_numpy()

    is_weekend = dat_user['is_weekend'].to_numpy()
    day = dat_user['day_norm'].to_numpy()
    week = dat_user['week_norm'].to_numpy() #TODO: check this
    decision_time = dat_user['DecisionTime'].to_numpy()
    # twice-daily walking: separate by decision slot (antic. affect outcome is daily; two treatment rows/day)
    ws_morning = WalkingSuggestion_lag1 * (1.0 - decision_time)
    ws_afternoon = WalkingSuggestion_lag1 * decision_time
    dow = dat_user['dow_norm'].to_numpy()
    dow_lag1 = dat_user['dow_norm_lag1'].to_numpy()
    
    

    # fill in missing values (NAN) with mean for predictors except for FourSC and Intercept
    yesterday_step_count = np.where(np.isnan(yesterday_step_count), np.nanmean(yesterday_step_count), yesterday_step_count)
    seven_day_step_count_avg = np.where(np.isnan(seven_day_step_count_avg), np.nanmean(seven_day_step_count_avg), seven_day_step_count_avg)
    prior2hour_step_count = np.where(np.isnan(prior2hour_step_count), np.nanmean(prior2hour_step_count), prior2hour_step_count)
    Previous7DaysRPA = np.where(np.isnan(Previous7DaysRPA), np.nanmean(Previous7DaysRPA), Previous7DaysRPA)
    recent_burden = np.where(np.isnan(recent_burden), np.nanmean(recent_burden), recent_burden)
    seven_day_pageview_count = np.where(np.isnan(seven_day_pageview_count), np.nanmean(seven_day_pageview_count), seven_day_pageview_count)
    past7days_daywearing = np.where(np.isnan(past7days_daywearing), np.nanmean(past7days_daywearing), past7days_daywearing)
    Interacted_7d_walk = np.where(np.isnan(Interacted_7d_walk), np.nanmean(Interacted_7d_walk), Interacted_7d_walk)
    Interacted_7d_salience = np.where(np.isnan(Interacted_7d_salience), np.nanmean(Interacted_7d_salience), Interacted_7d_salience)
    anticipated_affect = np.where(np.isnan(anticipated_affect), np.nanmean(anticipated_affect), anticipated_affect)
    anticipated_affect_yesterday = np.where(np.isnan(anticipated_affect_yesterday), np.nanmean(anticipated_affect_yesterday), anticipated_affect_yesterday)
    CAE_avg_lastweek = np.where(np.isnan(CAE_avg_lastweek), np.nanmean(CAE_avg_lastweek), CAE_avg_lastweek)
    perceived_utility_lastweek = np.where(np.isnan(perceived_utility_lastweek), np.nanmean(perceived_utility_lastweek), perceived_utility_lastweek)
    fourSC_lag1 = np.where(np.isnan(fourSC_lag1), np.nanmean(fourSC_lag1), fourSC_lag1)
    anticipated_affect_lag1 = np.where(np.isnan(anticipated_affect_lag1), np.nanmean(anticipated_affect_lag1), anticipated_affect_lag1)
    recorded_physical_activity_lag1 = np.where(np.isnan(recorded_physical_activity_lag1), np.nanmean(recorded_physical_activity_lag1), recorded_physical_activity_lag1)
    yesterday_pageview_count = np.where(np.isnan(yesterday_pageview_count), np.nanmean(yesterday_pageview_count), yesterday_pageview_count)
    hourly_pageview_count_lag1 = np.where(np.isnan(hourly_pageview_count_lag1), np.nanmean(hourly_pageview_count_lag1), hourly_pageview_count_lag1)
    yesterday_hourly_pageview_count = np.where(np.isnan(yesterday_hourly_pageview_count), np.nanmean(yesterday_hourly_pageview_count), yesterday_hourly_pageview_count)
    # yesterday_fitbitwearing_morning = np.where(np.isnan(yesterday_fitbitwearing_morning), np.nanmean(yesterday_fitbitwearing_morning), yesterday_fitbitwearing_morning)

    # fit M^Y_{w,d,t} model: 4 hour step count model
    fourSC_cond = np.stack([
        Intercept,
        fourSC_lag1,
        yesterday_step_count, seven_day_step_count_avg, 
        prior2hour_step_count, Previous7DaysRPA,
        recent_burden, seven_day_pageview_count, past7days_daywearing, 
        yesterday_planning_prompt, yesterday_SalienceMessage,
        Interacted_7d_walk, Interacted_7d_salience,
        anticipated_affect_yesterday,
        is_weekend, 
        dow, decision_time,
        CAE_avg_lastweek, perceived_utility_lastweek,
        WalkingSuggestion, WalkingSuggestion * yesterday_step_count,
        WalkingSuggestion * prior2hour_step_count,
        WalkingSuggestion * recent_burden,
        WalkingSuggestion * seven_day_pageview_count,
        WalkingSuggestion * past7days_daywearing,
        WalkingSuggestion * yesterday_planning_prompt,
        WalkingSuggestion * yesterday_SalienceMessage,
        WalkingSuggestion * Interacted_7d_walk,
        WalkingSuggestion * Interacted_7d_salience,
        WalkingSuggestion * anticipated_affect_yesterday,
        WalkingSuggestion * is_weekend,
        WalkingSuggestion * dow,
        WalkingSuggestion * decision_time,
        WalkingSuggestion * CAE_avg_lastweek,
        WalkingSuggestion * perceived_utility_lastweek
    ], axis=1)

    # check if there are any NaN values in the condition matrix
    if np.isnan(fourSC_cond).any():
        print(f"NaN values in condition matrix for user {userid}")
        print(np.isnan(fourSC_cond))

    # filter out rows where FourSC is NaN
    idx_obs_fourSC = ~np.isnan(fourSC)
    fourSC_cond_obs = fourSC_cond[idx_obs_fourSC, :]
    FourSC_obs = fourSC[idx_obs_fourSC]


    cv_fourSC = 5
    
    model_fourSC = RidgeCV(alphas=alpha_l2_list, fit_intercept=False, cv=cv_fourSC)
    model_fourSC.fit(fourSC_cond_obs, FourSC_obs)
    alpha_fourSC_l2 = model_fourSC.alpha_
    theta_fourSC_mean = model_fourSC.coef_
    pred_fourSC = model_fourSC.predict(fourSC_cond)
    resid_obs_fourSC = FourSC_obs - pred_fourSC[idx_obs_fourSC]
    # fill in the full residual array with NaN for unobserved
    resid_fourSC = np.full_like(fourSC, np.nan)
    resid_fourSC[idx_obs_fourSC] = resid_obs_fourSC
    sigma2_fourSC_mean = np.var(resid_obs_fourSC)

    # fit M^Y_{w,d} model: anticipated affect model
    # anticipated_affect_yesterday_norm ~ yesterday......
    # because this is at the end of the day, which corresponds to the next day's morning in RCT
    anticipated_affect_cond = np.stack([
        Intercept, 
        anticipated_affect_lag1,
        yesterday_step_count, recorded_physical_activity_lag1,
        yesterday_planning_prompt, yesterday_SalienceMessage,
        dow_lag1, 
        CAE_avg_lastweek, perceived_utility_lastweek,
        ws_morning, ws_afternoon,
        ws_morning * yesterday_step_count, ws_afternoon * yesterday_step_count,
        ws_morning * recorded_physical_activity_lag1, ws_afternoon * recorded_physical_activity_lag1,
        ws_morning * yesterday_planning_prompt, ws_afternoon * yesterday_planning_prompt,
        ws_morning * yesterday_SalienceMessage, ws_afternoon * yesterday_SalienceMessage,
        ws_morning * dow_lag1, ws_afternoon * dow_lag1,
        ws_morning * CAE_avg_lastweek, ws_afternoon * CAE_avg_lastweek,
        ws_morning * perceived_utility_lastweek, ws_afternoon * perceived_utility_lastweek
    ], axis=1)


    # Daily survey outcome: same Y on both decision rows → fit one row/day (here: morning decision only)
    idx_daily_antic = (decision_time == 0) & ~np.isnan(anticipated_affect_yesterday)
    cv_anticipated_affect = 5
    
    model_anticipated_affect = RidgeCV(alphas=alpha_l2_list, fit_intercept=False, cv=cv_anticipated_affect)
    model_anticipated_affect.fit(anticipated_affect_cond[idx_daily_antic], anticipated_affect_yesterday[idx_daily_antic])
    alpha_anticipated_affect_l2 = model_anticipated_affect.alpha_
    theta_anticipated_affect_mean = model_anticipated_affect.coef_
    # pred_anticipated_affect = np.full(len(anticipated_affect_yesterday_norm), np.nan)
    pred_anticipated_affect = model_anticipated_affect.predict(anticipated_affect_cond[idx_daily_antic])
    resid_obs_anticipated_affect = anticipated_affect_yesterday[idx_daily_antic] - pred_anticipated_affect


    # M^E_{w,d,t}
    # pageview~ yesterday......
    # because this is at the end of the day, which corresponds to the next day's morning in RCT
    fitbitwearing_morning_filled = np.where(np.isnan(fitbitwearing_morning), 0, fitbitwearing_morning)
    daily_present_filled = np.where(np.isnan(daily_present), 0, daily_present)
    pageview_cond = np.stack([
        Intercept, 
        hourly_pageview_count_lag1,
        fitbitwearing_morning_filled,
        daily_present_filled,
        recent_burden, 
        # seven_day_pageview_count, 
        # past7days_daywearing, 
        # yesterday_planning_prompt, yesterday_SalienceMessage,
        Interacted_7d_walk, Interacted_7d_salience,
        # anticipated_affect_yesterday_norm,
        # is_weekend, 
        dow, decision_time,
        CAE_avg_lastweek, perceived_utility_lastweek,
        WalkingSuggestion, 
        WalkingSuggestion * recent_burden,
        # WalkingSuggestion * yesterday_planning_prompt,
        # WalkingSuggestion * yesterday_SalienceMessage,
        # WalkingSuggestion * Interacted_7d_walk,
        # WalkingSuggestion * Interacted_7d_salience,
        # WalkingSuggestion * anticipated_affect_yesterday_norm,
        # WalkingSuggestion * is_weekend,
        WalkingSuggestion * dow,
        WalkingSuggestion * decision_time,
        WalkingSuggestion * CAE_avg_lastweek,
        WalkingSuggestion * perceived_utility_lastweek
    ], axis=1)

    # idx_daily_antic = (decision_time == 0) & ~np.isnan(HourlyPageviewCount_norm)
    # Adjust CV folds for view_status model
    cv_pageview = 5
    
    model_pageview= RidgeCV(alphas=alpha_l2_list, fit_intercept=False, cv=cv_pageview)
    model_pageview.fit(pageview_cond, HourlyPageviewCount)
    alpha_pageview_l2 = model_pageview.alpha_
    theta_pageview_mean = model_pageview.coef_
    pred_pageview = model_pageview.predict(pageview_cond)
    resid_obs_pageview = HourlyPageviewCount - pred_pageview


     # fit M^E_{w,d} model on daily rows (T/2; morning decision rows)
    idx_day = (decision_time == 0)

    _mu_pageview = np.nanmean(yesterday_hourly_pageview_count)
    pageview_filled = np.where(np.isnan(yesterday_hourly_pageview_count), _mu_pageview, yesterday_hourly_pageview_count)

    pageview_pair = pageview_filled.reshape(-1, 2)
    pageview_sw_morning = pageview_pair[:, 0]
    pageview_sw_afternoon = pageview_pair[:, 1]


    Intercept_day = Intercept[idx_day]
    yesterday_fitbitwearing_morning_day = yesterday_fitbitwearing_morning[idx_day]
    daily_present_filled_day = daily_present_filled[idx_day]
    recent_burden_day = recent_burden[idx_day]
    Interacted_7d_walk_day = Interacted_7d_walk[idx_day]
    Interacted_7d_salience_day = Interacted_7d_salience[idx_day]
    is_weekend_day = is_weekend[idx_day]
    dow_lag1_day = dow_lag1[idx_day]
    CAE_avg_lastweek_day = CAE_avg_lastweek[idx_day]
    perceived_utility_lastweek_day = perceived_utility_lastweek[idx_day]
    ws_morning_day = ws_morning[idx_day]
    ws_afternoon_day = ws_afternoon[idx_day]
    yesterday_present_day = yesterday_present[idx_day]
    fitbitwearing_morning_filled_day = fitbitwearing_morning_filled[idx_day]

    fitbitwearing_cond = np.stack([
        Intercept_day,
        yesterday_fitbitwearing_morning_day,
        pageview_sw_morning,
        pageview_sw_afternoon,
        daily_present_filled_day,
        # seven_day_pageview_count,
        recent_burden_day,
        # past7days_daywearing,
        # yesterday_planning_prompt, yesterday_SalienceMessage,
        Interacted_7d_walk_day, Interacted_7d_salience_day,
        # is_weekend_day,
        dow_lag1_day,
        CAE_avg_lastweek_day, perceived_utility_lastweek_day,
        ws_morning_day, ws_afternoon_day,
        # ws_morning * seven_day_pageview_count, ws_afternoon * seven_day_pageview_count,
        # ws_morning_day * yesterday_fitbitwearing_morning_day, ws_afternoon_day * yesterday_fitbitwearing_morning_day,
        ws_morning_day * recent_burden_day, ws_afternoon_day * recent_burden_day,
        # ws_morning * past7days_daywearing, ws_afternoon * past7days_daywearing,
        # ws_morning * yesterday_planning_prompt, ws_afternoon * yesterday_planning_prompt,
        # ws_morning * yesterday_SalienceMessage, ws_afternoon * yesterday_SalienceMessage,
        # ws_morning * Interacted_7d_walk, ws_afternoon * Interacted_7d_walk,
        # ws_morning_day * is_weekend_day, ws_afternoon_day * is_weekend_day,
        ws_morning_day * dow_lag1_day, ws_afternoon_day * dow_lag1_day,
        ws_morning_day * CAE_avg_lastweek_day, ws_afternoon_day * CAE_avg_lastweek_day,
        ws_morning_day * perceived_utility_lastweek_day, ws_afternoon_day * perceived_utility_lastweek_day
    ], axis=1)

    y_fitbit_day = fitbitwearing_morning[idx_day]
    idx_obs_fitbit = ~np.isnan(y_fitbit_day)
    # Adjust CV folds for view_status model
    cv_fitbitwearing = 5

    model_fitbitwearing= RidgeCV(alphas=alpha_l2_list, fit_intercept=False, cv=cv_fitbitwearing)
    model_fitbitwearing.fit(fitbitwearing_cond[idx_obs_fitbit], y_fitbit_day[idx_obs_fitbit])
    alpha_fitbitwearing_l2 = model_fitbitwearing.alpha_
    theta_fitbitwearing_mean = model_fitbitwearing.coef_
    pred_fitbitwearing = model_fitbitwearing.predict(fitbitwearing_cond[idx_obs_fitbit])
    resid_obs_fitbitwearing = y_fitbit_day[idx_obs_fitbit] - pred_fitbitwearing

    dailysurvey_cond = np.stack([
        Intercept_day,
        yesterday_present_day,
        fitbitwearing_morning_filled_day,
        pageview_sw_morning,
        pageview_sw_afternoon,
        # seven_day_pageview_count,
        recent_burden_day,
        # past7days_daywearing,
        # yesterday_planning_prompt, yesterday_SalienceMessage,
        Interacted_7d_walk_day, Interacted_7d_salience_day,
        # is_weekend_day,
        dow_lag1_day,
        CAE_avg_lastweek_day, perceived_utility_lastweek_day,
        ws_morning_day, ws_afternoon_day,
        # ws_morning_day * yesterday_present_day, ws_afternoon_day * yesterday_present_day,
        # ws_morning * seven_day_pageview_count, ws_afternoon * seven_day_pageview_count,
        ws_morning_day * recent_burden_day, ws_afternoon_day * recent_burden_day,
        # ws_morning * past7days_daywearing, ws_afternoon * past7days_daywearing,
        # ws_morning * yesterday_planning_prompt, ws_afternoon * yesterday_planning_prompt,
        # ws_morning * yesterday_SalienceMessage, ws_afternoon * yesterday_SalienceMessage,
        # ws_morning * Interacted_7d_walk, ws_afternoon * Interacted_7d_walk,
        # ws_morning_day * is_weekend_day, ws_afternoon_day * is_weekend_day,
        ws_morning_day * dow_lag1_day, ws_afternoon_day * dow_lag1_day,
        ws_morning_day * CAE_avg_lastweek_day, ws_afternoon_day * CAE_avg_lastweek_day,
        ws_morning_day * perceived_utility_lastweek_day, ws_afternoon_day * perceived_utility_lastweek_day
    ], axis=1)

    y_daily_day = daily_present[idx_day]
    idx_obs_daily = ~np.isnan(y_daily_day)
    # Adjust CV folds for view_status model
    cv_dailysurvey = 5

    model_dailysurvey= RidgeCV(alphas=alpha_l2_list, fit_intercept=False, cv=cv_dailysurvey)
    model_dailysurvey.fit(dailysurvey_cond[idx_obs_daily], y_daily_day[idx_obs_daily])
    alpha_dailysurvey_l2 = model_dailysurvey.alpha_
    theta_dailysurvey_mean = model_dailysurvey.coef_
    pred_dailysurvey = model_dailysurvey.predict(dailysurvey_cond[idx_obs_daily])
    resid_obs_dailysurvey = y_daily_day[idx_obs_daily] - pred_dailysurvey

    

    # Model Y_w
    # change the length of the condition matrix
    K = 14
    CAE_avg_sw = CAE_avg.reshape(-1, K)[:, 0]
    # perceived_utility_norm_sw = perceived_utility_norm.reshape(-1, K)[:, 0]
    CAE_avg_lastweek_sw = CAE_avg_lastweek.reshape(-1, K)[:, 0]
    week_sw = week.reshape(-1, K)[:, 0]
    Intercept_sw = np.ones(len(week_sw))
    
    # FourSC: 14 decision slots/week. Anticipated affect is daily (repeated across 2 decisions/day) → 7 columns/week
    foursc_wk = fourSC.reshape(-1, K)
    _mu_foursc = np.nanmean(fourSC)
    _mu_antic = np.nanmean(anticipated_affect)
    foursc_wk = np.where(np.isnan(foursc_wk), _mu_foursc, foursc_wk)
    _af = anticipated_affect.reshape(-1, K)
    antic_wk = np.nanmean(_af.reshape(-1, 7, 2), axis=2)
    antic_wk = np.where(np.isnan(antic_wk), _mu_antic, antic_wk)

    CAE_cond = np.hstack([
        Intercept_sw[:, None],
        CAE_avg_lastweek_sw[:, None],
        week_sw[:, None],
        foursc_wk,
        antic_wk
    ])

    ncv_w = 2
    
    idx_obs_CAE = ~np.isnan(CAE_avg_sw)
    CAE_cond_obs = CAE_cond[idx_obs_CAE, :]
    CAE_avg_sw_obs = CAE_avg_sw[idx_obs_CAE]
    n_obs_CAE = len(CAE_avg_sw_obs)
    # print(len(AA_avg_norm_sw_obs))

    if n_obs_CAE >= 2:
    # Use Generalized Cross-Validation (no explicit folds, avoids R^2 < 2 samples issue)
        model_CAE = RidgeCV(
            alphas=alpha_l2_list,
            fit_intercept=False,
            cv=None  # GCV
        )
        model_CAE.fit(CAE_cond_obs, CAE_avg_sw_obs)
        alpha_CAE_l2 = model_CAE.alpha_
    else:
        print(f'fallback to fixed alpha for CAE (Y_w) model for user {userid}')
        alpha_CAE_l2 = 1.0
        model_CAE = Ridge(alpha=alpha_CAE_l2, fit_intercept=False)
        model_CAE.fit(CAE_cond_obs, CAE_avg_sw_obs)
    
    

    
    theta_CAE_mean = model_CAE.coef_
    pred_CAE = model_CAE.predict(CAE_cond)
    resid_obs_CAE = CAE_avg_sw_obs - pred_CAE[idx_obs_CAE]
    resid_CAE = np.full_like(CAE_avg_sw, np.nan)
    resid_CAE[idx_obs_CAE] = resid_obs_CAE
    sigma2_CAE_mean = np.var(resid_obs_CAE)
    

    perceived_utility_sw = perceived_utility.reshape(-1, K)[:, 0]
    perceived_utility_lastweek_sw = perceived_utility_lastweek.reshape(-1, K)[:, 0]
    pw_sw = DailyPageviewCount.reshape(-1, K)
    _mu_pageview = np.nanmean(DailyPageviewCount)
    pw_sw = np.nanmean(pw_sw.reshape(-1, 7, 2), axis=2)
    pw_sw = np.where(np.isnan(pw_sw), _mu_pageview, pw_sw)
    dw_sw = DayWearing.reshape(-1, K)
    _mu_dw = np.nanmean(DayWearing)
    dw_sw = np.nanmean(dw_sw.reshape(-1, 7, 2), axis=2)
    dw_sw = np.where(np.isnan(dw_sw), _mu_dw, dw_sw)
    dp_sw = daily_present.reshape(-1, K)
    _mu_dp = np.nanmean(daily_present)
    dp_sw = np.nanmean(dp_sw.reshape(-1, 7, 2), axis=2)
    dp_sw = np.where(np.isnan(dp_sw), _mu_dp, dp_sw)
    
    

    # Model E_w
    # fit the perceived utility model (1D predictors as columns; pw/dw/dp are (n_weeks, 7) → use hstack not stack)
    perceived_utility_cond = np.hstack([
        Intercept_sw[:, None],
        perceived_utility_lastweek_sw[:, None],
        week_sw[:, None],
        pw_sw,
        dw_sw,
        dp_sw,
    ])
    
    idx_obs_perceived_utility = ~np.isnan(perceived_utility_sw)
    perceived_utility_cond_obs = perceived_utility_cond[idx_obs_perceived_utility, :]
    perceived_utility_sw_obs = perceived_utility_sw[idx_obs_perceived_utility]
    n_obs_perceived_utility = len(perceived_utility_sw_obs)
    
    if n_obs_perceived_utility >= 2:
    # Use Generalized Cross-Validation (no explicit folds, avoids R^2 < 2 samples issue)
        model_perceived_utility = RidgeCV(
            alphas=alpha_l2_list,
            fit_intercept=False,
            cv=None  # GCV
        )
        model_perceived_utility.fit(perceived_utility_cond_obs, perceived_utility_sw_obs)
        alpha_perceived_utility_l2 = model_perceived_utility.alpha_
    else:
        print(f'fallback to fixed alpha for perceived utility model for user {userid}')
        # n_obs_AA < 2 → you cannot really estimate a regression; use fixed alpha or skip
        alpha_perceived_utility_l2 = 1.0
        model_perceived_utility = Ridge(alpha=alpha_perceived_utility_l2, fit_intercept=False)
        model_perceived_utility.fit(perceived_utility_cond_obs, perceived_utility_sw_obs)


    
    theta_perceived_utility_mean = model_perceived_utility.coef_
    pred_perceived_utility = model_perceived_utility.predict(perceived_utility_cond)
    resid_obs_perceived_utility = perceived_utility_sw_obs - pred_perceived_utility[idx_obs_perceived_utility]
    resid_perceived_utility = np.full_like(perceived_utility_sw, np.nan)
    resid_perceived_utility[idx_obs_perceived_utility] = resid_obs_perceived_utility
    sigma2_perceived_utility_mean = np.var(resid_obs_perceived_utility)


    # fit the missingness model (emission of perceived utility)
    week_present_sw = week_present.reshape(-1, K)[:, 0]
    perceived_utility_sw_filled = np.where(np.isnan(perceived_utility_sw), np.nanmean(perceived_utility_sw), perceived_utility_sw)
    week_present_cond = np.stack([
        Intercept_sw,
        perceived_utility_sw_filled,
    ], axis=1)
    y_wp = np.asarray(week_present_sw, dtype=int).ravel()
    # Cs for L2 logistic: larger ridge alpha ~ stronger shrinkage ~ smaller C (heuristic: C ≈ 1/alpha)
    Cs_week_present = np.sort(1.0 / np.asarray(alpha_l2_list, dtype=float))
    if np.var(week_present_sw) > 0:
        n0, n1 = int(np.sum(y_wp == 0)), int(np.sum(y_wp == 1))
        min_class = min(n0, n1)
        cv_wp = min(ncv, min_class)
        if cv_wp >= 2:
            model_week_present = LogisticRegressionCV(
                Cs=Cs_week_present,
                cv=cv_wp,
                penalty="l2",
                solver="lbfgs",
                fit_intercept=False,
                scoring="neg_log_loss",
                max_iter=5000,
                random_state=seed,
            )
            model_week_present.fit(week_present_cond, y_wp)
            C_sel = float(model_week_present.C_[0])
        else:
            C_sel = float(Cs_week_present[len(Cs_week_present) // 2])
            model_week_present = LogisticRegression(
                penalty="l2",
                C=C_sel,
                solver="lbfgs",
                fit_intercept=False,
                max_iter=5000,
                random_state=seed,
            )
            model_week_present.fit(week_present_cond, y_wp)
        alpha_week_present_l2 = 1.0 / C_sel
        theta_week_present_mean = model_week_present.coef_.ravel()
        pred_week_present = model_week_present.predict_proba(week_present_cond)[:, 1]
        resid_obs_week_present = week_present_sw.astype(float) - pred_week_present
    else:
        print(f'the variance of week_present is 0 for user {userid}')
        const = np.nanmean(week_present_sw) if week_present_sw.size > 0 else 0.0
        alpha_week_present_l2 = float(alpha_l2_list[0])
        theta_week_present_mean = np.zeros(week_present_cond.shape[1])
        if const <= 0.0:
            theta_week_present_mean[0] = -25.0
        elif const >= 1.0:
            theta_week_present_mean[0] = 25.0
        else:
            theta_week_present_mean[0] = float(np.log(const / (1.0 - const)))
        pred_week_present = np.full_like(week_present_sw, const, dtype=float)
        resid_obs_week_present = np.full_like(week_present_sw, 0.0, dtype=float)

    # fit the emission of affective association

    # if j = 1, then the emission is 3 questions from CAE
    CAE_short_avg_sw = CAE_short_avg.reshape(-1, K)[:, 0]
    CAE_avg_sw_filled = np.where(np.isnan(CAE_avg_sw), np.nanmean(CAE_avg_sw), CAE_avg_sw)
    idx_obs_CAE_short_avg = ~np.isnan(CAE_short_avg_sw)
    CAE_short_avg_sw_obs = CAE_short_avg_sw[idx_obs_CAE_short_avg]
    if np.var(CAE_short_avg_sw_obs) > 0:
        CAE_short_avg_cond = np.stack([
            Intercept_sw,
            CAE_avg_sw_filled,
        ], axis=1)
        CAE_short_avg_cond_obs = CAE_short_avg_cond[idx_obs_CAE_short_avg, :]
        model_CAE_short_avg = RidgeCV(alphas=alpha_l2_list, fit_intercept=False, cv= None)
        model_CAE_short_avg.fit(CAE_short_avg_cond_obs, CAE_short_avg_sw_obs)
        alpha_CAE_short_avg_l2 = model_CAE_short_avg.alpha_
        theta_CAE_short_avg_mean = model_CAE_short_avg.coef_
        pred_CAE_short_avg = model_CAE_short_avg.predict(CAE_short_avg_cond)
        resid_obs_CAE_short_avg = CAE_short_avg_sw_obs - pred_CAE_short_avg[idx_obs_CAE_short_avg]
        resid_CAE_short_avg = np.full_like(CAE_short_avg_sw, np.nan)
        resid_CAE_short_avg[idx_obs_CAE_short_avg] = resid_obs_CAE_short_avg
        sigma2_CAE_short_avg_mean = np.var(resid_obs_CAE_short_avg)
    else:
        print(f'the variance of affective valuation is 0 for user {userid}')
        # constant = mean of observed values; if no observed, fall back to 0.0
        const = np.nanmean(CAE_short_avg_sw_obs) if CAE_short_avg_sw_obs.size > 0 else 0.0

        # alpha: just pick something valid so code downstream works
        alpha_AV_avg_l2 = alpha_l2_list[0]

        # coefficients: intercept = const, slope = 0
        theta_CAE_short_avg_mean = np.zeros(CAE_short_avg_cond.shape[1])
        theta_CAE_short_avg_mean[0] = const  # assuming first column is Intercept_sw

        # predictions: constant
        pred_CAE_short_avg = np.full_like(CAE_short_avg_sw, const)

        # residuals: 0 for observed, NaN for missing
        resid_CAE_short_avg = np.full_like(CAE_short_avg_sw, np.nan)
        resid_CAE_short_avg[idx_obs_CAE_short_avg] = 0.0

        # variance of residuals: zero
        sigma2_AV_avg_mean = 0.0
    


    alpha_cv.append([alpha_fourSC_l2, alpha_pageview_l2, alpha_fitbitwearing_l2, alpha_dailysurvey_l2, alpha_CAE_l2, alpha_perceived_utility_l2, alpha_week_present_l2, alpha_CAE_short_avg_l2])

    theta_pageview_list.append(theta_pageview_mean)
    theta_fitbitwearing_list.append(theta_fitbitwearing_mean)
    theta_eodcomplete_list.append(theta_dailysurvey_mean)
    digits = 3
    env_para = {
        'theta_fourSC': np.round(theta_fourSC_mean, digits).tolist(),
        'theta_antic': np.round(theta_anticipated_affect_mean, digits).tolist(),
        'theta_pageview': np.round(theta_pageview_mean, digits).tolist(),
        'theta_fitbitwearing': np.round(theta_fitbitwearing_mean, digits).tolist(),
        'theta_dailysurvey': np.round(theta_dailysurvey_mean, digits).tolist(),
        'theta_CAE': np.round(theta_CAE_mean, digits).tolist(),
        'theta_perceived_utility': np.round(theta_perceived_utility_mean, digits).tolist(),
        'theta_week_present': np.round(theta_week_present_mean, digits).tolist(),
        'theta_CAE_short_avg': np.round(theta_CAE_short_avg_mean, digits).tolist(),
        'resid_fourSC': np.round(resid_fourSC, digits).tolist(),
        'resid_antic': np.round(resid_obs_anticipated_affect, digits).tolist(),
        'resid_pageview': np.round(resid_obs_pageview, digits).tolist(),
        'resid_fitbitwearing': np.round(resid_obs_fitbitwearing, digits).tolist(),
        'resid_dailysurvey': np.round(resid_obs_dailysurvey, digits).tolist(),
        'resid_CAE': np.round(resid_obs_CAE, digits).tolist(),
        'resid_perceived_utility': np.round(resid_obs_perceived_utility, digits).tolist(),
        'resid_week_present': np.round(resid_obs_week_present, digits).tolist(),
        'resid_CAE_short_avg': np.round(resid_CAE_short_avg, digits).tolist(),
    }
    predicted = {
        'pred_fourSC': np.round(pred_fourSC, digits).tolist(),
        'pred_antic': np.round(pred_anticipated_affect, digits).tolist(),
        'pred_pageview': np.round(pred_pageview, digits).tolist(),
        'pred_fitbitwearing': np.round(pred_fitbitwearing, digits).tolist(),
        'pred_dailysurvey': np.round(pred_dailysurvey, digits).tolist(),
        'pred_CAE': np.round(pred_CAE, digits).tolist(),
        'pred_perceived_utility': np.round(pred_perceived_utility, digits).tolist(),
        'pred_week_present': np.round(pred_week_present, digits).tolist(),
        'pred_CAE_short_avg': np.round(pred_CAE_short_avg, digits).tolist(),
    }

    with open(file_params_env_prefix + str(userid) + '.json', 'w') as file:
        json.dump(env_para, file)
    with open(file_pred_prefix + str(userid) + '.json', 'w') as file:
        json.dump(predicted, file)

    
np.savetxt(file_user_ids, userid_all, fmt='%d')
alpha_cv = np.array(alpha_cv)
    

fallback to fixed alpha for CAE (Y_w) model for user 151
fallback to fixed alpha for perceived utility model for user 151
the variance of affective valuation is 0 for user 151
the variance of affective valuation is 0 for user 160
the variance of week_present is 0 for user 195
the variance of week_present is 0 for user 204
the variance of week_present is 0 for user 225


In [30]:
# impute query effects

xi = 1/8

# Convert list of arrays to numpy array
theta_pageview_list = np.array(theta_pageview_list)
theta_fitbitwearing_list = np.array(theta_fitbitwearing_list)
theta_eodcomplete_list = np.array(theta_eodcomplete_list)

# step 1: take population average of theta_perceived_utility
theta_pageview_mean = np.mean(theta_pageview_list, axis=0)
theta_fitbitwearing_mean = np.mean(theta_fitbitwearing_list, axis=0)
theta_eodcomplete_mean = np.mean(theta_eodcomplete_list, axis=0)

# step 2: scale by 1/8
theta_pageview_mean = theta_pageview_mean * xi
theta_fitbitwearing_mean = theta_fitbitwearing_mean * xi
theta_eodcomplete_mean = theta_eodcomplete_mean * xi

# replace the intercept para with 2 * the average of the rest paras
theta_pageview_mean[0] = 2 * np.mean(theta_pageview_mean[1:])
theta_fitbitwearing_mean[0] = 2 * np.mean(theta_fitbitwearing_mean[1:])
theta_eodcomplete_mean[0] = 2 * np.mean(theta_eodcomplete_mean[1:])

# scale all participants' perceived utility by the population average
theta_pageview_list = theta_pageview_list * xi
theta_fitbitwearing_list = theta_fitbitwearing_list * xi
theta_eodcomplete_list = theta_eodcomplete_list * xi

# calculate the variance
theta_pageview_variance = np.var(theta_pageview_list, axis=0)
theta_fitbitwearing_variance = np.var(theta_fitbitwearing_list, axis=0)
theta_eodcomplete_variance = np.var(theta_eodcomplete_list, axis=0)

# calculate the variance of the intercept by 2 * the average of the rest variances
theta_pageview_variance[0] = 2 * np.mean(theta_pageview_variance[1:])
theta_fitbitwearing_variance[0] = 2 * np.mean(theta_fitbitwearing_variance[1:])
theta_eodcomplete_variance[0] = 2 * np.mean(theta_eodcomplete_variance[1:])

# sample participants' perceived utility from the population average
theta_pageview_sampled = np.random.normal(theta_pageview_mean, np.sqrt(theta_pageview_variance), size=(len(userid_all), len(theta_pageview_mean)))
theta_fitbitwearing_sampled = np.random.normal(theta_fitbitwearing_mean, np.sqrt(theta_fitbitwearing_variance), size=(len(userid_all), len(theta_fitbitwearing_mean)))
theta_eodcomplete_sampled = np.random.normal(theta_eodcomplete_mean, np.sqrt(theta_eodcomplete_variance), size=(len(userid_all), len(theta_eodcomplete_mean)))


digits = 3  # same as before

for i, userid in enumerate(userid_all):
    # Load existing env params
    with open(file_params_env_prefix + str(userid) + '.json', 'r') as f:
        env_para = json.load(f)

    # Existing theta_perceived_utility (original coefficients)
    old_theta_pageview = np.array(env_para.get('theta_pageview', []), dtype=float)
    old_theta_fitbitwearing = np.array(env_para.get('theta_fitbitwearing', []), dtype=float)
    old_theta_eodcomplete = np.array(env_para.get('theta_eodcomplete', []), dtype=float)

    # New sampled coefficients for this user
    new_theta_pageview = np.round(theta_pageview_sampled[i, :], digits)
    new_theta_fitbitwearing = np.round(theta_fitbitwearing_sampled[i, :], digits)
    new_theta_eodcomplete = np.round(theta_eodcomplete_sampled[i, :], digits)

    # select certain coefficients
    new_theta_pageview = new_theta_pageview[[0, 4, 7, 8, 9, 10]]
    new_theta_fitbitwearing = new_theta_fitbitwearing[[0, 5, 8, 9, 10]]
    new_theta_eodcomplete = new_theta_eodcomplete[[0, 5, 8, 9, 10]]

    # Assign sign to the new coefficients according to domain knowledge
    # these parameters are the query effects there is I_w before them
    new_theta_pageview[0] = -1 * np.abs(new_theta_pageview[0])
    new_theta_pageview[1] = -1 * np.abs(new_theta_pageview[1])
    new_theta_pageview[2] = -1 * np.abs(new_theta_pageview[2])
    new_theta_pageview[3] = -1 * np.abs(new_theta_pageview[3])
    new_theta_pageview[4] = 1 * np.abs(new_theta_pageview[4])
    new_theta_pageview[5] = 1 * np.abs(new_theta_pageview[5])
    
    new_theta_fitbitwearing[0] = -1 * np.abs(new_theta_fitbitwearing[0])
    new_theta_fitbitwearing[1] = -1 * np.abs(new_theta_fitbitwearing[1])
    new_theta_fitbitwearing[2] = -1 * np.abs(new_theta_fitbitwearing[2])
    new_theta_fitbitwearing[3] = 1 * np.abs(new_theta_fitbitwearing[3])
    new_theta_fitbitwearing[4] = 1 * np.abs(new_theta_fitbitwearing[4])
 
    new_theta_eodcomplete[0] = -1 * np.abs(new_theta_eodcomplete[0])
    new_theta_eodcomplete[1] = -1 * np.abs(new_theta_eodcomplete[1])
    new_theta_eodcomplete[2] = -1 * np.abs(new_theta_eodcomplete[2])
    new_theta_eodcomplete[3] = 1 * np.abs(new_theta_eodcomplete[3])
    new_theta_eodcomplete[4] = 1 * np.abs(new_theta_eodcomplete[4])


    # Concatenate original + sampled
    theta_concat_pageview = np.concatenate([old_theta_pageview, new_theta_pageview])
    theta_concat_fitbitwearing = np.concatenate([old_theta_fitbitwearing, new_theta_fitbitwearing])
    theta_concat_eodcomplete = np.concatenate([old_theta_eodcomplete, new_theta_eodcomplete])

    # Store back as list
    env_para['theta_pageview'] = theta_concat_pageview.tolist()
    env_para['theta_fitbitwearing'] = theta_concat_fitbitwearing.tolist()
    env_para['theta_dailysurvey'] = theta_concat_eodcomplete.tolist()

    # Save updated env file
    with open(file_params_env_prefix + str(userid) + '.json', 'w') as f:
        json.dump(env_para, f)


